# Bab 9: Memodelkan Probabilitas dan Non-Linearitas: Fungsi Aktivasi
*(Modeling Probabilities and Nonlinearities: Activation Functions)*

## Eksplorasi Teoritis Mendalam (Theoretical Deep-Dive)

Pada bab-bab sebelumnya, kita telah memahami bahwa jaringan saraf adalah serangkaian perkalian matriks. Namun, dunia nyata tidak selalu bekerja dalam garis lurus (linear). Jika Anda ingin memprediksi harga rumah, kurvanya mungkin melengkung. Jika Anda ingin memprediksi gambar kucing atau anjing, Anda memerlukan keluaran berupa probabilitas (0% hingga 100%). Bab ini memperkenalkan **Fungsi Aktivasi Non-Linear** untuk memodelkan fenomena tersebut.

### 1. Masalah Linearitas (The Problem with Linearity)
Jika kita hanya menumpuk matriks bobot tanpa henti, secara matematis hasilnya tetap akan menjadi satu garis lurus. Sebuah garis lurus tidak bisa memecahkan masalah yang kompleks (seperti memisahkan titik-titik data yang melingkar). Kita butuh "bengkokan". Di sinilah fungsi aktivasi masuk. Mereka membengkokkan ruang matematika sehingga jaringan saraf dapat melingkari dan mengelompokkan data yang sangat kompleks.

### 2. Membentuk Probabilitas: Sigmoid dan Tanh
*   **Sigmoid:** Fungsi ini mengambil angka apa pun (baik itu -1000 atau +1000) dan "memerasnya" (squash) menjadi rentang antara **0 dan 1**. Ini sangat ideal untuk prediksi probabilitas ganda (binary). Jika outputnya 0.9, berarti mesin yakin 90% bahwa jawabannya adalah "Benar".
*   **Tanh (Tangens Hiperbolik):** Mirip dengan Sigmoid, tetapi memeras angka menjadi rentang antara **-1 dan 1**. Tanh sangat bagus digunakan di *hidden layers* (lapisan tersembunyi) karena menjaga rata-rata data tetap berada di angka 0, yang membantu *Gradient Descent* bekerja lebih stabil dan cepat.

### 3. Klasifikasi Banyak Kelas: Softmax
Jika kita ingin mesin membedakan angka tulisan tangan dari 0 hingga 9 (10 kelas berbeda), kita tidak bisa hanya menggunakan Sigmoid. Mengapa? Karena Sigmoid memprediksi probabilitas secara independen. Bisa saja mesin berkata: "Ini 90% angka 2, tapi juga 80% angka 3". Itu tidak masuk akal dalam probabilitas dunia nyata.
*   **Softmax:** Fungsi ini memastikan bahwa **total probabilitas dari seluruh kelas yang ada selalu sama dengan 1 (atau 100%)**.
*   **Mekanisme:** Jika jaringan meningkatkan tebakan probabilitas untuk "Angka 2", maka Softmax akan secara otomatis menekan probabilitas untuk angka-angka lainnya. Ini menciptakan persaingan (*competitive learning*) antar kelas, membuat tebakan mesin jauh lebih tajam dan meyakinkan.

In [ ]:
import numpy as np

# =====================================================================
# REPRODUKSI KODE BAB 9: FUNGSI AKTIVASI (SIGMOID, TANH, SOFTMAX)
# Mendemonstrasikan bagaimana fungsi ini mengubah data dan cara turunannya
# =====================================================================

np.random.seed(1)

# 1. FUNGSI SIGMOID & TURUNANNYA
# Memeras angka menjadi rentang 0 hingga 1
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    # Asumsi: input x sudah melewati fungsi sigmoid
    return x * (1 - x)

# 2. FUNGSI TANH & TURUNANNYA
# Memeras angka menjadi rentang -1 hingga 1
def tanh(x):
    return np.tanh(x)

def tanh_deriv(x):
    # Asumsi: input x sudah melewati fungsi tanh
    return 1 - (x ** 2)

# 3. FUNGSI SOFTMAX
# Mengubah sekumpulan angka tebakan menjadi probabilitas baku (Total = 100%)
def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

# --- DEMONSTRASI ARSITEKTUR KLASIFIKASI ---
print("Demonstrasi Forward Propagation dengan Tanh dan Softmax:")
print("-" * 60)

# Simulasi 1 baris gambar yang memiliki 3 fitur pixel
layer_0 = np.array([[1.0, 0.5, 0.2]])

# Bobot Layer 1 (3 input -> 4 hidden nodes)
weights_0_1 = np.random.rand(3, 4) - 0.5
# Bobot Layer 2 (4 hidden nodes -> 3 kelas klasifikasi: Kucing, Anjing, Burung)
weights_1_2 = np.random.rand(4, 3) - 0.5

# Forward Propagation dengan Tanh di Hidden Layer
layer_1 = tanh(np.dot(layer_0, weights_0_1))
print(f"Hasil Hidden Layer (diperas Tanh antara -1 dan 1):\n{layer_1}\n")

# Forward Propagation dengan Softmax di Final Output
layer_2_raw = np.dot(layer_1, weights_1_2)
print(f"Hasil Tebakan Mentah (Raw Output):\n{layer_2_raw}\n")

layer_2_prob = softmax(layer_2_raw)
print(f"Hasil Probabilitas Softmax (Persentase Kucing, Anjing, Burung):")
print(layer_2_prob)

# Membuktikan bahwa total probabilitas Softmax = 1 (100%)
total_prob = np.sum(layer_2_prob)
print(f"\nTotal Keseluruhan Probabilitas Softmax: {total_prob:.2f} (100%)")

### Penjelasan Eksekusi Kode:

1.  **Fungsi `np.exp(x)`:** Ini adalah konstanta Euler ($e \approx 2.718$). Penggunaan fungsi eksponensial di `sigmoid` dan `softmax` memastikan bahwa angka negatif pun akan diubah menjadi positif (karena probabilitas tidak boleh bernilai negatif).
2.  **`sigmoid_deriv` dan `tanh_deriv`:** Ini adalah operasi turunan (kalkulus) yang digunakan saat Backpropagation. Keunggulan matematika dari fungsi aktivasi ini adalah turunannya sangat mudah dihitung. Misalnya, turunan dari `tanh` hanyalah `1 - output_sebelumnya_kuadrat`. Ini menghemat daya komputasi secara masif.
3.  **Mekanisme Softmax (`temp / np.sum(...)`)**: Di blok Softmax, kita mengambil hasil eksponensial dari setiap tebakan mentah (raw score), lalu membaginya dengan total keseluruhan tebakan tersebut. Inilah yang mengubah skor bebas menjadi "Porsi Probabilitas". Jika skor mentah untuk "Anjing" sangat tinggi, porsi probabilitasnya akan mendekati 1, secara matematis memaksa probabilitas untuk "Kucing" dan "Burung" menjadi mendekati 0.
4.  **Hasil Output:** Dapat dilihat di output eksekusi bahwa tebakan mentah (yang bisa bernilai negatif atau melampaui 1) berhasil dijinakkan oleh `Softmax` menjadi `[0.32, 0.40, 0.26]`, yang jika dijumlahkan nilainya pasti pas `1.00` atau `100%`.